# Chapter 4:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/modern-recommender-systems/blob/main/notebooks/chapter-04/precision_metrics.ipynb)

This notebook introduces precision metrics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

np.random.seed(42)
num_users = 500
num_products = 100
n_recommend = 10
test_items_per_user = 2

# Simulate users
def simulate_user(test_items, n_rec, rec_quality=0.2):
    """
    rec_quality is the probability that a test item appears in the recommendations (controls average precision)
    """
    true_test = set(np.random.choice(range(num_products), size=test_items, replace=False))
    # recommend n_rec items, with 'rec_quality' chance of including each test item
    recs = set(np.random.choice(range(num_products), size=n_rec, replace=False))
    for t in true_test:
        if np.random.rand() < rec_quality:
            # Add the test item in recommendation with certain probability
            recs.add(t)
    # Ensure we don't exceed n_rec recommendations
    recs = set(list(recs)[:n_rec])
    hits = len(recs & true_test)
    precision = hits / n_rec
    recall = hits / test_items
    return precision, recall

# Run simulation for different "qualities"
qualities = [0.0, 0.2, 0.4, 0.6]
results = []
for q in qualities:
    precisions = []
    for _ in range(num_users):
        precision, recall = simulate_user(test_items_per_user, n_recommend, rec_quality=q)
        precisions.append(precision)
    results.append(precisions)

# Visualize
fig, axes = plt.subplots(1, len(qualities), figsize=(18, 4), sharey=True)

for idx, (prec_list, q) in enumerate(zip(results, qualities)):
    ax = axes[idx]
    sns.histplot(prec_list, kde=False, bins=np.linspace(0, 1, 12), ax=ax, color='green')
    ax.set_title(f'rec_quality={q}\nMean precision={np.mean(prec_list):.2f}')
    ax.set_xlabel('Precision per user')
    ax.set_ylabel('User count')
    ax.axvline(np.mean(prec_list), color='red', linestyle='--', label='Mean')
    ax.legend()

plt.suptitle('Distribution of Precision across Users for different "rec_qualities"\n(Decent average may still hide many users with 0 precision)')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
num_users = 500
target_mean_precision = 0.2

# 1. Uniform scenario
prec_uniform = np.random.normal(loc=target_mean_precision, scale=0.03, size=num_users)
prec_uniform = np.clip(prec_uniform, 0, 1)

# 2. Biased scenario: only some users are lucky
# Let exactly target_mean_precision fraction have precision=1, rest precision=0
num_lucky = int(np.round(target_mean_precision * num_users))
prec_biased = np.zeros(num_users)
lucky_users = np.random.choice(range(num_users), size=num_lucky, replace=False)
prec_biased[lucky_users] = 1.0

print(f"Uniform mean precision: {prec_uniform.mean():.2f}")
print(f"Biased mean precision:  {prec_biased.mean():.2f}")

# Plot
plt.figure(figsize=(12,5))
sns.histplot(prec_uniform, bins=np.linspace(0, 1, 12), kde=False, label='Uniform', color='blue')
sns.histplot(prec_biased, bins=np.linspace(0, 1, 12), kde=False, label='Biased', color='green', alpha=0.6)
plt.axvline(prec_uniform.mean(), color="blue", linestyle="--", label="Uniform Mean")
plt.axvline(prec_biased.mean(), color="red", linestyle="--", label="Biased Mean")
plt.legend()
plt.title("Same mean precision, very different user distributions!")
plt.xlabel("Precision per user")
plt.ylabel("User count")
plt.show()